# Model Training and Comparison

In this notebook, we'll train and compare three distinct recommendation architectures on our dataset:
1. **User-Based KNN (Memory-based CF)**
2. **Singular Value Decomposition (Matrix Factorization)**
3. **Neural Collaborative Filtering (Deep Learning)**

In [ ]:
import sys
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import accuracy
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

sys.path.append(os.path.abspath('..'))
import config
from src.data_loader import load_processed
from src.preprocessing import sample_data, create_train_test_split, build_surprise_dataset
from src.models.knn_cf import UserBasedCF
from src.models.svd_model import SVDRecommender
from src.models.neural_cf import NeuralCFRecommender

sns.set_style('whitegrid')
import warnings
warnings.filterwarnings('ignore')

## 1. Data Preparation

We need to sample the data (to keep training times manageable for the notebook) and prepare the train/test splits. We also need to build the `surprise` dataset structures for KNN and SVD, and label-encoded structures for NCF.

In [ ]:
data_path = os.path.join('..', config.PROCESSED_DATA_DIR, 'all_ratings.parquet')
df = load_processed(data_path)

sampled_df = sample_data(df)
train_df, test_df = create_train_test_split(sampled_df, strategy='random')

# Surprise Datasets
train_data_surp = build_surprise_dataset(train_df).build_full_trainset()
test_data_surp = build_surprise_dataset(test_df).build_full_trainset().build_testset()

# Encoders for NCF
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

user_encoder.fit(sampled_df['user_id'])
item_encoder.fit(sampled_df['movie_id'])

train_ncf_df = train_df.copy()
train_ncf_df['user_idx'] = user_encoder.transform(train_df['user_id'])
train_ncf_df['item_idx'] = item_encoder.transform(train_df['movie_id'])

test_ncf_df = test_df.copy()
test_ncf_df['user_idx'] = user_encoder.transform(test_df['user_id'])
test_ncf_df['item_idx'] = item_encoder.transform(test_df['movie_id'])

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)
print(f"NCF Vocab Size - Users: {n_users}, Items: {n_items}")

## 2. Train SVD Baseline

Matrix Factorization is robust against sparsity and relatively fast to train.

In [ ]:
svd_model = SVDRecommender(n_factors=config.SVD_PARAMS['n_factors'], 
                           n_epochs=config.SVD_PARAMS['n_epochs'],
                           lr_all=config.SVD_PARAMS['lr_all'],
                           reg_all=config.SVD_PARAMS['reg_all'])

# TODO: try svd_model.tune() later to optimize factors
start_time = time.time()
svd_model.fit(train_data_surp)
svd_train_time = time.time() - start_time

svd_preds = svd_model.algo.test(test_data_surp)
svd_rmse = accuracy.rmse(svd_preds, verbose=False)
svd_mae = accuracy.mae(svd_preds, verbose=False)

print(f"SVD RMSE: {svd_rmse:.4f}")

## 3. Train KNN Baseline

Memory-based models are computationally intensive because they calculate similarities across many vectors. We'll stick to User-based CF here.

In [ ]:
knn_model = UserBasedCF(k=config.KNN_USER_PARAMS['k'], 
                        sim_name=config.KNN_USER_PARAMS['sim_options']['name'])

start_time = time.time()
knn_model.fit(train_data_surp)
knn_train_time = time.time() - start_time

knn_preds = knn_model.algo.test(test_data_surp)
knn_rmse = accuracy.rmse(knn_preds, verbose=False)
knn_mae = accuracy.mae(knn_preds, verbose=False)

print(f"KNN RMSE: {knn_rmse:.4f}")

## 4. Train Neural CF

Deep learning can model non-linear user-item interactions. We'll use the NeuMF architecture combining Generalized Matrix Factorization and Multi-Layer Perceptron.

In [ ]:
ncf_model.model.eval()
test_preds = []
VAL_BATCH = 4096
import torch
with torch.no_grad():
    for start in range(0, len(test_ncf_df), VAL_BATCH):
        end = start + VAL_BATCH
        val_u = torch.tensor(test_ncf_df['user_idx'].values[start:end], dtype=torch.long).to(ncf_model.device)
        val_i = torch.tensor(test_ncf_df['item_idx'].values[start:end], dtype=torch.long).to(ncf_model.device)
        test_preds.extend(ncf_model.model(val_u, val_i).cpu().numpy())
test_preds = np.clip(np.array(test_preds), 1.0, 5.0)
test_actuals = test_ncf_df['rating'].values
ncf_rmse = np.sqrt(mean_squared_error(test_actuals, test_preds))
ncf_mae = mean_absolute_error(test_actuals, test_preds)
print(f'NCF Final RMSE: {ncf_rmse:.4f}')


### Neural CF Loss Curve
Let's see how our PyTorch model converged over the epochs.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(ncf_model.train_losses, marker='o', color='purple')
plt.title("Neural CF Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.xticks(range(len(ncf_model.train_losses)))
plt.show()

## 5. Model Comparison

Let's put all the metrics side by side to see how they stack up against each other.

In [ ]:
results = pd.DataFrame([
    {
        'Model': 'SVD (Matrix Factorization)',
        'RMSE': svd_rmse,
        'MAE': svd_mae,
        'Train Time (s)': round(svd_train_time, 2),
        'Notes': 'Excellent baseline, fast, handles sparsity well'
    },
    {
        'Model': 'User-Based KNN',
        'RMSE': knn_rmse,
        'MAE': knn_mae,
        'Train Time (s)': round(knn_train_time, 2),
        'Notes': 'Slow prediction, memory intensive, interpretable'
    },
    {
        'Model': 'Neural CF (NeuMF)',
        'RMSE': ncf_rmse,
        'MAE': ncf_mae,
        'Train Time (s)': round(ncf_train_time, 2),
        'Notes': 'Non-linear interactions, GPU-accelerated, complex to tune'
    }
])

display(results.set_index('Model'))

## 6. Real-World Output: Top 10 Recommendations

Metrics are great, but what do the actual recommendations look like? Let's take a sample user and compare what each model suggests for them.

In [ ]:
# Grab a random user who has enough ratings
sample_user = train_df['user_id'].value_counts().index[0]
all_items = sampled_df['movie_id'].unique()
user_rated = train_df[train_df['user_id'] == sample_user]['movie_id'].tolist()

print(f"Top 10 Recommendations for User {sample_user}\n")

svd_top = svd_model.get_top_n(sample_user, all_items, n=10, exclude_rated=True)
knn_top = knn_model.get_top_n(sample_user, all_items, n=10, exclude_rated=True)
ncf_top = ncf_model.get_top_n(sample_user, all_items, n=10, exclude_rated=True, user_rated_items=user_rated)

print(f"{'SVD ID':<15} | {'KNN ID':<15} | {'NCF ID':<15}")
print("-" * 50)
for i in range(10):
    svd_item = svd_top[i][0] if i < len(svd_top) else '-'
    knn_item = knn_top[i][0] if i < len(knn_top) else '-'
    ncf_item = ncf_top[i][0] if i < len(ncf_top) else '-'
    print(f"{svd_item:<15} | {knn_item:<15} | {ncf_item:<15}")

## Trade-offs and Discussion

* **Accuracy vs. Complexity:** SVD provides an extremely strong baseline that is difficult to beat without significant hyperparameter tuning on the Neural CF model. NCF has the potential to capture complex non-linear interactions, but requires careful regularization to avoid overfitting.
* **Speed vs. Interpretability:** KNN models are highly interpretable (we can see *which* similar users drove a recommendation) but they are notoriously slow at inference time since they must calculate distances against many neighbors. SVD and NCF are essentially instant lookups/forward passes at inference time.
* **Cold-Start Handling:** None of these models inherently handle pure cold-start scenarios well (a completely new user with 0 ratings). However, matrix factorization can default to user/item biases, whereas KNN fails completely. To truly solve cold-start, we would need content-based features (e.g., movie genres, user demographics) which could be easily appended to the NCF model architecture.